# TIGER: sample 10,000 rows and save a fixed 6,400/1,600/2,000 split

Place this notebook in the root of the cloned `tiger` repository.

It loads:

`data-processed/off-target.bz2`

Then it:

1. reproduces TIGER's default non-indel guide-type filter;
2. computes `observed_lfc = mean(lfc_r1, lfc_r2, lfc_r3)`;
3. keeps aligned guide–target pairs;
4. uses the official online TIGER context:
   - 3 nt of 5′ context;
   - 0 nt of 3′ context;
5. randomly samples exactly 10,000 rows with seed 42;
6. creates:
   - 6,400 training samples;
   - 1,600 validation samples;
   - 2,000 unseen samples;
7. saves all exact indices, split tables, aligned TXT files, and checksums.

You do not need to delete older files manually. This notebook writes to a new output folder:

`results/tiger_repository_sampled_10000_<activity_mode>/`


In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_FILE = Path("data-processed/off-target.bz2")

ACTIVITY_MODE = "observed_lfc"
# ACTIVITY_MODE = "negative_observed_lfc"

TOTAL_SAMPLED = 10000
TRAIN_SIZE = 6400
VALIDATION_SIZE = 1600
UNSEEN_SIZE = 2000

SAMPLE_SEED = 42
SPLIT_SEED = 42

GUIDE_LENGTH = 23
CONTEXT_5P = 3
CONTEXT_3P = 0

OUTPUT_DIR = Path(
    f"results/tiger_repository_sampled_10000_{ACTIVITY_MODE}"
)
FULL_EXPORT_DIR = OUTPUT_DIR / "sampled_full_dataset"
SPLIT_TABLE_DIR = OUTPUT_DIR / "saved_splits"
SPLIT_EXPORT_DIR = OUTPUT_DIR / "split_sequences"

for directory in [
    FULL_EXPORT_DIR,
    SPLIT_TABLE_DIR,
    SPLIT_EXPORT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

assert DATA_FILE.exists(), f"Missing {DATA_FILE.resolve()}"

assert TRAIN_SIZE + VALIDATION_SIZE + UNSEEN_SIZE == TOTAL_SAMPLED

print("Dataset:", DATA_FILE.resolve())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
raw_data = pd.read_pickle(DATA_FILE)

required_columns = {
    "guide_seq",
    "target_seq",
    "5p_context",
    "3p_context",
    "guide_type",
    "lfc_r1",
    "lfc_r2",
    "lfc_r3",
}

missing_columns = required_columns - set(raw_data.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

allowed_guide_types = {
    "PM",
    "SM",
    "DM",
    "RDM",
    "TM",
    "RTM",
}

data = raw_data.loc[
    raw_data["guide_type"].isin(allowed_guide_types)
].copy()

replicate_columns = ["lfc_r1", "lfc_r2", "lfc_r3"]

for column in replicate_columns:
    data[column] = pd.to_numeric(
        data[column],
        errors="coerce",
    )

data["observed_lfc"] = data[replicate_columns].mean(axis=1)

data = data.dropna(
    subset=[
        "guide_seq",
        "target_seq",
        "5p_context",
        "3p_context",
        "observed_lfc",
    ]
).copy()


def clean_sequence(series):
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .str.replace("U", "T", regex=False)
    )


data["guide_sequence"] = clean_sequence(data["guide_seq"])
data["target_sequence"] = clean_sequence(data["target_seq"])

raw_5p_context = clean_sequence(data["5p_context"])
raw_3p_context = clean_sequence(data["3p_context"])

if ACTIVITY_MODE == "observed_lfc":
    data["activity"] = data["observed_lfc"]
elif ACTIVITY_MODE == "negative_observed_lfc":
    data["activity"] = -data["observed_lfc"]
else:
    raise ValueError(
        "ACTIVITY_MODE must be 'observed_lfc' or "
        "'negative_observed_lfc'."
    )

valid_rows = (
    data["guide_sequence"].str.fullmatch("[ACGT]+")
    & data["target_sequence"].str.fullmatch("[ACGT]+")
    & raw_5p_context.str.fullmatch("[ACGT]*")
    & raw_3p_context.str.fullmatch("[ACGT]*")
    & (
        data["guide_sequence"].str.len()
        == data["target_sequence"].str.len()
    )
)

data = data.loc[valid_rows].reset_index(drop=True)
raw_5p_context = raw_5p_context.loc[valid_rows].reset_index(drop=True)

data["5p_context"] = raw_5p_context.str.slice(-CONTEXT_5P, None)
data["3p_context"] = ""

data["source_row_id"] = np.arange(len(data), dtype=int)

if len(data) < TOTAL_SAMPLED:
    raise ValueError(
        f"Need at least {TOTAL_SAMPLED} filtered rows, found {len(data)}."
    )

print("Filtered repository rows:", len(data))
print("Guide types:")
print(data["guide_type"].value_counts())
print("Sequence lengths:")
print(data["guide_sequence"].str.len().value_counts().sort_index())


## Randomly sample exactly 10,000 rows

In [ ]:
sampled_data = data.sample(
    n=TOTAL_SAMPLED,
    replace=False,
    random_state=SAMPLE_SEED,
).copy()

sampled_data = sampled_data.reset_index(drop=True)
sampled_data["sample_id"] = np.arange(TOTAL_SAMPLED, dtype=int)

print("Sampled rows:", len(sampled_data))
print("Activity range:", sampled_data["activity"].min(), "to", sampled_data["activity"].max())

display(
    sampled_data[
        [
            "sample_id",
            "source_row_id",
            "guide_sequence",
            "target_sequence",
            "5p_context",
            "3p_context",
            "activity",
        ]
    ].head()
)


## Export the complete sampled 10,000-row dataset

In [ ]:
full_export_mapping = {
    "guide_sequences.txt": sampled_data["guide_sequence"],
    "target_sequences.txt": sampled_data["target_sequence"],
    "5p_context.txt": sampled_data["5p_context"],
    "3p_context.txt": sampled_data["3p_context"],
    "activity.txt": sampled_data["activity"],
}

for filename, series in full_export_mapping.items():
    series.to_csv(
        FULL_EXPORT_DIR / filename,
        index=False,
        header=False,
    )

metadata_columns = [
    column
    for column in [
        "sample_id",
        "source_row_id",
        "gene",
        "guide_id",
        "guide_type",
        "guide_sequence",
        "target_sequence",
        "5p_context",
        "3p_context",
        "lfc_r1",
        "lfc_r2",
        "lfc_r3",
        "observed_lfc",
        "activity",
        "target_fold",
        "guide_fold",
    ]
    if column in sampled_data.columns
]

sampled_data[metadata_columns].to_csv(
    FULL_EXPORT_DIR / "complete_sampled_dataset.tsv",
    sep="\t",
    index=False,
)

print("Exported sampled 10,000-row dataset.")


## Create the exact 6,400/1,600/2,000 split

First, exactly 2,000 rows are selected as unseen. The remaining 8,000 rows
are split into exactly 6,400 training and 1,600 validation rows.


In [ ]:
seen_data, unseen_data = train_test_split(
    sampled_data,
    test_size=UNSEEN_SIZE,
    random_state=SPLIT_SEED,
    shuffle=True,
)

train_data, validation_data = train_test_split(
    seen_data,
    test_size=VALIDATION_SIZE,
    random_state=SPLIT_SEED,
    shuffle=True,
)

train_data = train_data.reset_index(drop=True)
validation_data = validation_data.reset_index(drop=True)
unseen_data = unseen_data.reset_index(drop=True)

assert len(train_data) == TRAIN_SIZE
assert len(validation_data) == VALIDATION_SIZE
assert len(unseen_data) == UNSEEN_SIZE

train_ids = set(train_data["sample_id"])
validation_ids = set(validation_data["sample_id"])
unseen_ids = set(unseen_data["sample_id"])

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(unseen_ids)
assert validation_ids.isdisjoint(unseen_ids)

display(
    pd.DataFrame(
        {
            "subset": ["train", "validation", "unseen"],
            "n_samples": [
                len(train_data),
                len(validation_data),
                len(unseen_data),
            ],
            "fraction_of_sampled_10000": [0.64, 0.16, 0.20],
        }
    )
)


## Save exact split tables, indices, and TXT files

In [ ]:
train_file = SPLIT_TABLE_DIR / "train_split.csv"
validation_file = SPLIT_TABLE_DIR / "validation_split.csv"
unseen_file = SPLIT_TABLE_DIR / "unseen_split.csv"

train_data.to_csv(train_file, index=False)
validation_data.to_csv(validation_file, index=False)
unseen_data.to_csv(unseen_file, index=False)

np.savetxt(
    OUTPUT_DIR / "sampled_source_row_ids.txt",
    sampled_data["source_row_id"].to_numpy(dtype=int),
    fmt="%d",
)

np.savetxt(
    OUTPUT_DIR / "train_indices_within_sampled_10000.txt",
    train_data["sample_id"].to_numpy(dtype=int),
    fmt="%d",
)

np.savetxt(
    OUTPUT_DIR / "validation_indices_within_sampled_10000.txt",
    validation_data["sample_id"].to_numpy(dtype=int),
    fmt="%d",
)

np.savetxt(
    OUTPUT_DIR / "unseen_indices_within_sampled_10000.txt",
    unseen_data["sample_id"].to_numpy(dtype=int),
    fmt="%d",
)


def export_subset(frame, subset_name):
    subset_dir = SPLIT_EXPORT_DIR / subset_name
    subset_dir.mkdir(parents=True, exist_ok=True)

    mapping = {
        f"{subset_name}_guide_sequences.txt": frame["guide_sequence"],
        f"{subset_name}_target_sequences.txt": frame["target_sequence"],
        f"{subset_name}_5p_context.txt": frame["5p_context"],
        f"{subset_name}_3p_context.txt": frame["3p_context"],
        f"{subset_name}_activity.txt": frame["activity"],
    }

    for filename, series in mapping.items():
        series.to_csv(
            subset_dir / filename,
            index=False,
            header=False,
        )

    frame[metadata_columns].to_csv(
        subset_dir / f"{subset_name}_complete.tsv",
        sep="\t",
        index=False,
    )


for subset_name, subset_frame in [
    ("train", train_data),
    ("validation", validation_data),
    ("unseen", unseen_data),
]:
    export_subset(subset_frame, subset_name)

print("Saved exact split files.")


## Save checksums and reproducibility settings

In [ ]:
def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


manifest = {
    "activity_mode": ACTIVITY_MODE,
    "activity_definition": (
        "mean(lfc_r1, lfc_r2, lfc_r3)"
        if ACTIVITY_MODE == "observed_lfc"
        else "-mean(lfc_r1, lfc_r2, lfc_r3)"
    ),
    "sample_seed": SAMPLE_SEED,
    "split_seed": SPLIT_SEED,
    "total_sampled": TOTAL_SAMPLED,
    "n_train": TRAIN_SIZE,
    "n_validation": VALIDATION_SIZE,
    "n_unseen": UNSEEN_SIZE,
    "official_context_5p": CONTEXT_5P,
    "official_context_3p": CONTEXT_3P,
    "allowed_guide_types": sorted(allowed_guide_types),
    "source_dataset_sha256": sha256(DATA_FILE),
    "sampled_guide_sha256": sha256(
        FULL_EXPORT_DIR / "guide_sequences.txt"
    ),
    "sampled_target_sha256": sha256(
        FULL_EXPORT_DIR / "target_sequences.txt"
    ),
    "sampled_5p_context_sha256": sha256(
        FULL_EXPORT_DIR / "5p_context.txt"
    ),
    "sampled_3p_context_sha256": sha256(
        FULL_EXPORT_DIR / "3p_context.txt"
    ),
    "sampled_activity_sha256": sha256(
        FULL_EXPORT_DIR / "activity.txt"
    ),
    "train_sha256": sha256(train_file),
    "validation_sha256": sha256(validation_file),
    "unseen_sha256": sha256(unseen_file),
}

with open(
    OUTPUT_DIR / "split_manifest.json",
    "w",
) as handle:
    json.dump(manifest, handle, indent=2)

print("Manifest saved to:", (OUTPUT_DIR / "split_manifest.json").resolve())
